# SUNy Fine-Tuning with Unsloth + Google Colab

**Goal:** Fine-tune open-weight models to think and act like SUNy — an autonomous coding agent that follows a 5-stage pipeline (INTENT_PARSE → PLAN → EXECUTION → VERIFICATION → FINALIZE), makes tool calls, self-corrects on failures, and provides friendly plain-English summaries.

| Section | What it does |
|---------|-------------|
| **1. Setup** | Installs Unsloth, connects Google Drive, installs dependencies |
| **2. Load Base Model** | Loads Qwen3.5 9B (or Llama 3.1 8B) with 4-bit QLoRA |
| **3. Load Data** | Loads the synthetic SUNy training data in ShareGPT JSONL format |
| **4. Format Data** | Converts to Unsloth's chat template format |
| **5. Train** | Runs LoRA fine-tuning with optimized hyperparameters |
| **6. Save** | Saves LoRA adapter locally and uploads to Hugging Face Hub |
| **7. Test** | Runs inference tests to verify SUNy behavior |
| **8. Deploy** | Instructions to deploy via OpenRouter / vLLM / Ollama |

---

## Prerequisites

- ✅ A Google account (for Colab + Drive)
- ✅ A Hugging Face account (for model upload) — [Sign up free](https://huggingface.co/join)
- ✅ Your training data JSONL file (see `scripts/generate-suny-training-data.py`)
- ✅ *(Optional)* A Together AI or OpenRouter account for managed deployment

---

## Choose Your Base Model

Unsloth supports many models. For SUNy, the best options are:

| Model | Size | Why SUNy? | Colab T4? |
|-------|------|-----------|-----------|
| **Qwen3.5 9B** *(recommended)* | 9B params | Excellent coding + instruction following. Best SUNy persona retention. | ✅ Yes (QLoRA) |
| **Llama 3.1 8B** | 8B params | Solid general-purpose. Slightly weaker at tool-use patterns. | ✅ Yes (QLoRA) |
| **DeepSeek Coder V2 Lite** | 16B params | Great at code. Larger → slower on Colab. | ⚠️ Marginal |
| **Mistral 7B** | 7B params | Fastest training. Weaker at complex multi-step reasoning. | ✅ Yes |

> **Recommendation:** Start with **Qwen3.5 9B**. It has strong coding ability and responds well to the structured 5-stage format.

---

---
## Step 1: Setup Environment

Run this cell to install Unsloth and all dependencies.

In [ ]:
# @title 1.1 Install Unsloth
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# Install Unsloth with all dependencies
!pip install -U pip
!pip install unsloth
!pip install --upgrade --no-deps unsloth

# Verify install
import unsloth
print(f'Unsloth version: {unsloth.__version__}')

In [ ]:
# @title 1.2 Mount Google Drive (to store data + models)
from google.colab import drive
drive.mount('/content/drive')

# Create working directories
!mkdir -p /content/drive/MyDrive/suny-training/data
!mkdir -p /content/drive/MyDrive/suny-training/models
!mkdir -p /content/drive/MyDrive/suny-training/logs
print('\nDirectories ready at: /content/drive/MyDrive/suny-training/')

In [ ]:
# @title 1.3 Upload or Copy Training Data
import shutil
from pathlib import Path

TRAIN_DATA_SOURCE = '/content/drive/MyDrive/suny-training/data/suny-training-data.jsonl'
TRAIN_DATA_DEST  = '/content/suny-training-data.jsonl'

# Check if data already exists in Drive
if os.path.exists(TRAIN_DATA_SOURCE):
    shutil.copy(TRAIN_DATA_SOURCE, TRAIN_DATA_DEST)
    print(f'Copied from Drive: {TRAIN_DATA_SOURCE}')
elif os.path.exists(TRAIN_DATA_DEST):
    print(f'Already at: {TRAIN_DATA_DEST}')
else:
    # Upload widget
    from google.colab import files
    print('Please upload your suny-training-data.jsonl file:')
    uploaded = files.upload()
    for fn in uploaded.keys():
        shutil.move(fn, TRAIN_DATA_DEST)
        # Also save to Drive
        shutil.copy(TRAIN_DATA_DEST, TRAIN_DATA_SOURCE)
        print(f'Saved: {TRAIN_DATA_DEST}')

# Count lines
with open(TRAIN_DATA_DEST, 'r') as f:
    line_count = sum(1 for _ in f)
print(f'\nTraining examples: {line_count}')

---
## Step 2: Load Base Model with QLoRA

This loads the model in 4-bit quantization and attaches LoRA adapters. Total VRAM usage: ~8-10GB on a T4 (16GB).

In [ ]:
# @title 2.1 Select and Load Base Model
import torch
from unsloth import FastLanguageModel

# ── CONFIGURE YOUR MODEL HERE ────────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen3.5-9B-Instruct"  # Recommended primary choice
# Alternatives:
# "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"   # Llama 3.1 8B
# "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"     # Mistral 7B
# "unsloth/deepseek-coder-v2-lite-instruct"       # DeepSeek Coder (larger)
# ────────────────────────────────────────────────────────────────────────

max_seq_length = 4096  # SUNy conversations can be long. 4096 covers most.
dtype = None           # Auto-detect: Float16 for T4, BFloat16 for A100
load_in_4bit = True    # 4-bit QLoRA — enables training on T4 16GB

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    device_map="auto",
)

print(f'Model loaded: {MODEL_NAME}')
print(f'Model size: {sum(p.numel() for p in model.parameters())/1e9:.1f}B parameters')

In [ ]:
# @title 2.2 Configure LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,                  # LoRA rank. 16 = good balance of quality + speed
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],                     # All linear layers — best for instruction tuning
    lora_alpha=32,         # Scaling factor. Higher = stronger adaptation
    lora_dropout=0.05,     # Small dropout for generalization
    bias="none",          # Don't train bias params (standard LoRA practice)
    use_gradient_checkpointing=True,    # Saves VRAM at cost of slightly slower training
    random_state=42,       # Reproducibility
    use_rslora=False,      # Rank-stabilized LoRA (optional, can help)
    loftq_config=None,     # LoftQ quantization (not needed with 4-bit base)
)

print('LoRA adapters attached.')
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M')

---
## Step 3: Load and Format Training Data

The data is in **ShareGPT JSONL format**:
```json
{"conversations": [
  {"from": "human", "value": "Add a search bar..."},
  {"from": "gpt", "value": "<reasoning>...</reasoning>\n<tool_call>...</tool_call>..."}
]}
```

Unsloth's `get_chat_template` function converts this to the model's native chat format automatically.

In [ ]:
# @title 3.1 Load Chat Template
from unsloth.chat_templates import get_chat_template

# Unsloth auto-detects the correct chat template for your model
tokenizer = get_chat_template(
    tokenizer,
    chat_template="chatml",  # Qwen uses ChatML format
    # For Llama:
    # chat_template="llama-3.1",
)

print(f'Chat template: {tokenizer.chat_template[:80]}...')

In [ ]:
# @title 3.2 Load Dataset from JSONL
import json
from datasets import Dataset

def load_sharegpt_jsonl(path):
    """Load ShareGPT-format JSONL into a HuggingFace Dataset."""
    data = []
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                entry = json.loads(line)
                if 'conversations' in entry:
                    data.append(entry)
            except json.JSONDecodeError:
                continue
    return Dataset.from_list(data)

dataset = load_sharegpt_jsonl(TRAIN_DATA_DEST)
print(f'Dataset loaded: {len(dataset)} examples')
print(f'Columns: {dataset.column_names}')
print(f'\nFirst example type: {dataset[0]["task_type"]}')
print(f'First example complexity: {dataset[0]["complexity"]}')

In [ ]:
# @title 3.3 Format Dataset for Training
from unsloth.chat_templates import standardize_sharegpt
from unsloth.chat_templates import get_chat_template

# Standardize: convert ShareGPT format to messages format
dataset = standardize_sharegpt(dataset)
print(f'Standardized: {len(dataset)} examples')

# Apply chat template to format each conversation
def apply_template(examples):
    texts = []
    for messages in examples['conversations']:
        # Tokenize with the chat template
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        texts.append(text)
    return {'text': texts}

dataset = dataset.map(apply_template, batched=True)
print(f'\nFormatted example (truncated):')
print(dataset[0]['text'][:300] + '...')

In [ ]:
# @title 3.4 Train/Validation Split
dataset = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset = dataset['train']
eval_dataset = dataset['test']

print(f'Training examples:   {len(train_dataset)}')
print(f'Validation examples: {len(eval_dataset)}')

---
## Step 4: Fine-Tune

Key hyperparameters:
- **Learning rate:** 2e-4 (standard for LoRA)
- **Batch size:** 2 (max for T4 16GB with QLoRA + gradient checkpointing)
- **Gradient accumulation:** 4 (effective batch = 8)
- **Epochs:** 1-3 (SUNy data is synthetic — 2 epochs is usually enough)
- **Optimizer:** AdamW 8-bit (saves VRAM)
- **Warmup:** 10% of steps

In [ ]:
# @title 4.1 Configure Training Arguments
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# ── CONFIGURE TRAINING ───────────────────────────────────────────────────
NUM_EPOCHS = 2           # 1-3. 2 is a good start. Increase if loss plateaus.
LEARNING_RATE = 2e-4     # Standard LoRA LR. 1e-4 for more conservative.
PER_DEVICE_BATCH = 2     # 2 fits T4 16GB with 4K sequence length
GRADIENT_ACCUM = 4       # Effective batch = 2 * 4 = 8
MAX_STEPS = -1           # -1 = let epochs control the length
WARMUP_RATIO = 0.1       # 10% warmup
# ────────────────────────────────────────────────────────────────────────

training_args = TrainingArguments(
    output_dir='/content/suny-lora-checkpoints',
    num_train_epochs=NUM_EPOCHS,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRADIENT_ACCUM,
    warmup_ratio=WARMUP_RATIO,
    learning_rate=LEARNING_RATE,
    fp16=not is_bfloat16_supported(),  # T4 → fp16, A100 → bf16
    bf16=is_bfloat16_supported(),
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=0.1,       # Evaluate every 10% of training
    save_strategy="steps",
    save_steps=0.25,      # Save checkpoint every 25% of training
    save_total_limit=2,   # Keep only 2 latest checkpoints
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    optim="adamw_8bit",  # Saves VRAM
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    report_to="none",    # No wandb/MLflow needed for Colab
    dataloader_num_workers=2,
)

print('Training arguments ready.')
print(f'Effective batch size: {PER_DEVICE_BATCH * GRADIENT_ACCUM}')
print(f'Total steps: ~{len(train_dataset) // (PER_DEVICE_BATCH * GRADIENT_ACCUM) * NUM_EPOCHS}')

In [ ]:
# @title 4.2 Create Trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field='text',
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,  # Packing speeds up training on short sequences
    args=training_args,
)

print('Trainer created. Ready to train!')

In [ ]:
# @title 4.3 Start Training!
print('Training started...')
print(f'Using device: {model.device}')

# Show VRAM before training
gpu_stats = torch.cuda.get_device_properties(0)
start_vram = torch.cuda.memory_reserved(0) / 1024**3
print(f'GPU: {gpu_stats.name}')
print(f'VRAM reserved: {start_vram:.2f} GB / {gpu_stats.total_memory / 1024**3:.2f} GB')

# ⚡ TRAIN ⚡
trainer_stats = trainer.train()

# Show VRAM after training
end_vram = torch.cuda.memory_reserved(0) / 1024**3
print(f'\nTraining complete!')
print(f'VRAM before: {start_vram:.2f} GB')
print(f'VRAM after:  {end_vram:.2f} GB')

In [ ]:
# @title 4.4 Plot Training Loss (Optional)
import matplotlib.pyplot as plt

logs = trainer.state.log_history
train_loss = [x['loss'] for x in logs if 'loss' in x]
eval_loss = [x['eval_loss'] for x in logs if 'eval_loss' in x]

plt.figure(figsize=(10, 4))
if train_loss:
    plt.subplot(1, 2, 1)
    plt.plot(train_loss)
    plt.title('Training Loss')
    plt.xlabel('Step')
    plt.ylabel('Loss')
if eval_loss:
    plt.subplot(1, 2, 2)
    plt.plot(eval_loss)
    plt.title('Validation Loss')
    plt.xlabel('Step')
    plt.ylabel('Loss')
plt.tight_layout()
plt.show()

if eval_loss:
    print(f'Final eval loss: {eval_loss[-1]:.4f}')
    print(f'Best eval loss:  {min(eval_loss):.4f}')

---
## Step 5: Save the Fine-Tuned Model

Save the LoRA adapter (just the changed weights — ~33MB) and optionally merge + save the full model.

In [ ]:
# @title 5.1 Save LoRA Adapter Locally + to Google Drive
import shutil

# Local save
model.save_pretrained('/content/suny-lora-final')
tokenizer.save_pretrained('/content/suny-lora-final')

# Copy to Google Drive
drive_path = '/content/drive/MyDrive/suny-training/models/suny-lora-final'
if os.path.exists(drive_path):
    shutil.rmtree(drive_path)
shutil.copytree('/content/suny-lora-final', drive_path)

print('LoRA adapter saved!')
print(f'  Local:  /content/suny-lora-final')
print(f'  Drive:  {drive_path}')

# Show size
total_size = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fn in os.walk('/content/suny-lora-final') for f in fn)
print(f'  Size:   {total_size / 1024**2:.1f} MB')

### Upload to Hugging Face Hub

Uploading to Hugging Face makes your model available anywhere — OpenRouter, Together AI, vLLM, Ollama, or direct inference.

Run this cell and enter your Hugging Face token when prompted.

In [ ]:
# @title 5.2 Login to Hugging Face Hub
from huggingface_hub import notebook_login, HfApi

# Login
notebook_login()

# Verify
api = HfApi()
user = api.whoami()
print(f'Logged in as: {user["name"]}')

In [ ]:
# @title 5.3 Push LoRA Adapter to Hugging Face Hub
# ── CONFIGURE YOUR REPO NAME ─────────────────────────────────────────────
HF_USERNAME = "your-username"  # ← CHANGE THIS to your Hugging Face username
MODEL_REPO_NAME = "SUNy-Qwen3.5-9B-LoRA"  # ← Or choose your own name
# ────────────────────────────────────────────────────────────────────────

repo_id = f'{HF_USERNAME}/{MODEL_REPO_NAME}'

model.push_to_hub(
    repo_id,
    tokenizer=tokenizer,
    private=True,  # Private by default. Set to False for public.
    use_auth_token=True,
)

print(f'\n✅ Model uploaded!')
print(f'   https://huggingface.co/{repo_id}')
print(f'\nNext steps:')
print(f'   1. Go to the repo and set it to "public" if you want others to use it')
print(f'   2. Use on Together AI, OpenRouter, or local inference')

### Optional: Save Merged Model (16-bit)

For deployment on vLLM or Together AI, you need the full merged model (not just the LoRA adapter).
This merges LoRA weights into the base model and saves in 16-bit.

> ⏳ This takes ~5-10 minutes and produces an ~16GB model.

In [ ]:
# @title 5.4 (Optional) Merge and Save Full Model in 16-bit
# ── CONFIGURE ─────────────────────────────────────────────────────────────
MERGE_AND_SAVE = False  # Set to True to merge and save full model
# ────────────────────────────────────────────────────────────────────────

if MERGE_AND_SAVE:
    print('Merging LoRA weights into base model...')
    model.save_pretrained_merged(
        '/content/suny-merged-16bit',
        tokenizer,
        save_method='merged_16bit',
    )

    # Copy to Drive
    drive_merged = '/content/drive/MyDrive/suny-training/models/suny-merged-16bit'
    if os.path.exists(drive_merged):
        shutil.rmtree(drive_merged)
    shutil.copytree('/content/suny-merged-16bit', drive_merged)

    print(f'Full model saved!')
    print(f'  Local:  /content/suny-merged-16bit')
    print(f'  Drive:  {drive_merged}')

    # Push merged to Hugging Face
    if HF_USERNAME and HF_USERNAME != 'your-username':
        merged_repo_id = f'{HF_USERNAME}/{MODEL_REPO_NAME}-merged'
        model.push_to_hub_merged(
            merged_repo_id,
            tokenizer,
            save_method='merged_16bit',
            private=True,
            use_auth_token=True,
        )
        print(f'  HuggingFace: https://huggingface.co/{merged_repo_id}')
else:
    print('Merge skipped. Set MERGE_AND_SAVE = True to produce a full merged model.')

---
## Step 6: Test Inference

Let's verify the fine-tuned model produces SUNy-like responses.

In [ ]:
# @title 6.1 Test the SUNy Fine-Tuned Model
from unsloth.chat_templates import get_chat_template

# Ensure we're using the fine-tuned model for inference
FastLanguageModel.for_inference(model)

test_prompts = [
    "Add a search bar to the dashboard that filters users by name and email.",
    "Fix the error: the login form crashes when the user types special characters.",
    "What model are you?",  # SUNy should NOT reveal it's an AI model
]

for prompt in test_prompts:
    print('=' * 60)
    print(f'USER: {prompt}')
    print()

    messages = [
        {'role': 'user', 'content': prompt},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors='pt',
    ).to('cuda')

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=1024,
        temperature=0.7,
        top_p=0.9,
        repetition_penalty=1.05,
        do_sample=True,
    )

    response = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print(f'SUNy: {response[:600]}')
    print()

print('=' * 60)
print('Inference tests complete!')

In [ ]:
# @title 6.2 SUNy Behavior Checklist
print('Check if the model shows SUNy behavior:')
print()
checks = [
    '❓ Does it start with INTENT_PARSE / reading files?'   ,
    '❓ Does it produce a PLAN before writing code?'        ,
    '❓ Does it make tool calls (read_file, write_file, etc.)?',
    '❓ Does it VERIFY with lint/tests after writing?'      ,
    '❓ Does it FINALIZE with a plain-English summary?'     ,
    '❓ Does it refuse to reveal model/provider names?'     ,
    '❓ Does it auto-correct when checks fail?'             ,
]
for c in checks:
    print(c)
print()
print('If most are ✅, your model has learned SUNy behavior!')

---
## Step 7: Deployment

Your fine-tuned SUNy model is ready. Here's how to use it in production.

### Option A: OpenRouter (Easiest)

OpenRouter supports custom Hugging Face models. Add the model ID you uploaded above.

1. Go to [openrouter.ai/models](https://openrouter.ai/models)
2. Click **"Add Custom Model"**
3. Enter: `your-username/SUNy-Qwen3.5-9B-LoRA` (or the merged version)
4. Set pricing (cheap: $0.10/M input tokens)
5. In SUNy's Admin → API Keys, add an OpenRouter key with `model_id_override` set to your custom model

**Cost estimate:** OpenRouter charges ~$0.05-0.15/M tokens for custom HF models + their 10% markup.

### Option B: Together AI (Cheapest Managed)

Together AI offers fine-tuning + inference for ~$0.48/1M tokens.

1. Go to [together.ai](https://together.ai) → **Fine-tuning**
2. Upload your JSONL training data
3. Select Qwen3.5 9B as base model
4. Set: `lora_r=16`, `lora_alpha=32`, `learning_rate=2e-4`, `epochs=2`
5. Deploy the endpoint when done
6. Use the endpoint in SUNy's Admin → API Keys as an OpenAI-compatible provider

**Cost estimate:** ~$1.50 total for fine-tuning + $0.48/1M tokens for inference.

### Option C: Self-Hosted vLLM (Free but Needs GPU)

If you have a GPU machine (even a used RTX 3090 for ~$700):

```bash
# Install vLLM
pip install vllm

# Serve the merged model
python -m vllm.entrypoints.openai.api_server \
    --model your-username/SUNy-Qwen3.5-9B-LoRA-merged \
    --trust-remote-code \
    --port 8000

# In SUNy's Admin, add an OpenAI-compatible API key:
#   Provider:  OpenAI-compatible
#   Base URL:  http://your-server:8000/v1
#   Model:     your-username/SUNy-Qwen3.5-9B-LoRA-merged
```

### Option D: Ollama (Simplest Local Setup)

```bash
# Install Ollama: https://ollama.ai
ollama pull qwen:9b

# Create a Modelfile
echo "FROM qwen:9b" > Modelfile
echo "SYSTEM \"You are SUNy, an autonomous coding assistant...\"" >> Modelfile

# Create the model with system prompt
ollama create SUNy -f Modelfile

# Run it
ollama run SUNy
```

> Note: This doesn't use your fine-tuned weights — just the system prompt. For true fine-tuned inference via Ollama, use [llama.cpp](https://github.com/ggerganov/llama.cpp) with the merged model converted to GGUF.

### Step 8: Integrate into SUNy

Once deployed, configure SUNy to use your fine-tuned model:

1. Go to **SUNy Admin → API Keys**
2. Add a new key with:
   - **Provider:** The provider you're using (OpenRouter, Together, or OpenAI-compatible)
   - **Mode:** `fast` or `pro` (your choice)
   - **Model Override:** Your model ID (e.g., `your-username/SUNy-Qwen3.5-9B-LoRA`)
3. Save and test in a new chat

The fine-tuned model will now respond as SUNy — following the 5-stage pipeline, making tool calls, self-correcting, and providing friendly summaries — without needing the full system prompt.

---
## Tips for Better Results

### If the model doesn't show SUNy behavior:
- **Train longer:** Increase to 3 epochs
- **More data:** Generate more examples (run the generator with `--count 3000`)
- **Higher LoRA rank:** Try `r=32` instead of `r=16`
- **Lower learning rate:** Try `1e-4` for more conservative training

### If the model is too verbose or too short:
- **Max sequence length:** Increase to 6144 if conversations are being truncated
- **Temperature:** Lower to 0.5 for more focused responses

### If training is too slow on T4:
- **Reduce sequence length:** 2048 instead of 4096 (if your data fits)
- **Increase gradient accumulation:** 8 instead of 4
- **Reduce LoRA rank:** `r=8` instead of `r=16`

---

## Summary

| Task | Status |
|------|--------|
| ✅ Generate synthetic data | `python scripts/generate-suny-training-data.py --count 1200` |
| ✅ Fine-tune on Colab | This notebook — ~3-4 hours on T4 |
| ✅ Upload to Hugging Face | Done (LoRA adapter ~33MB) |
| ✅ Deploy to OpenRouter | ~$0.10/M tokens |
| ✅ Configure in SUNy | Add API key with model override |

Your SUNy instance now has a custom-tailored model that understands the 5-stage pipeline, tool-calling patterns, and autonomous coding behavior — trained without needing any real user history.

---

*Happy building! 🚀*